[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/06_residual_and_connections.ipynb)

# 06. Residual and connection structures — residual, LayerScale, HC, and paper-faithful mHC

이 노트북은 residual connection을 단순한 `x + F(x)`에서 시작해 multi-stream Hyper-Connections와 mHC까지 연결한다.

이번 버전의 기준은 **모델 크기만 줄이고 mHC의 계산 구조는 유지하는 것**이다. 따라서 mHC의 dynamic maps는 작은 learnable scale `alpha`, static bias, RMSNorm, 그리고 `sigmoid / 2*sigmoid / Sinkhorn-Knopp` 제약을 모두 포함한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Plain residual connection

single-stream residual은 다음 update다.

`x_next = x + F(x)`

identity path가 branch를 우회하므로 signal과 gradient가 직접 흐를 수 있다.


In [ ]:
x = torch.randn(2, 8, device=device)

branch = nn.Sequential(
    nn.Linear(8, 16),
    nn.SiLU(),
    nn.Linear(16, 8),
).to(device)

residual_output = x + branch(x)

print("input:", x.shape)
print("residual output:", residual_output.shape)


## 2. ReZero and LayerScale

ReZero는 residual branch 전체에 learned scalar를 곱하고, LayerScale은 channel별 learned scale을 곱한다. 둘 다 residual branch의 초기 크기를 작게 유지하는 방법이다.


In [ ]:
rezero_alpha = nn.Parameter(
    torch.zeros(1, device=device)
)

layer_scale_gamma = nn.Parameter(
    1e-4 * torch.ones(8, device=device)
)

rezero_output = x + rezero_alpha * branch(x)
layerscale_output = x + layer_scale_gamma * branch(x)

print(
    "ReZero initial change:",
    (rezero_output - x).abs().max().item(),
)
print(
    "LayerScale initial change:",
    (layerscale_output - x).abs().max().item(),
)


## 3. Hyper-Connections: one branch, multiple residual streams

Residual state를 `n`개의 parallel streams로 확장하면 한 layer는 세 mapping을 사용한다.

- `H_pre`: 여러 residual streams를 branch input 하나로 읽는다.
- `H_res`: residual streams끼리 섞는다.
- `H_post`: branch output을 여러 residual streams에 다시 쓴다.

핵심 update는

`X_next = H_res X + H_post^T F(H_pre X)`

이다.


In [ ]:
batch_size = 2
num_streams = 3
channels = 8

streams = torch.randn(
    batch_size,
    num_streams,
    channels,
    device=device,
)

H_pre = torch.tensor(
    [0.8, 0.4, 0.2],
    device=device,
)

H_post = torch.tensor(
    [1.0, 0.6, 0.3],
    device=device,
)

H_res = torch.tensor(
    [
        [0.8, 0.1, 0.1],
        [0.1, 0.8, 0.1],
        [0.1, 0.1, 0.8],
    ],
    device=device,
)

branch_input = torch.einsum(
    "s,bsc->bc",
    H_pre,
    streams,
)

branch_output = branch(branch_input)

residual_mixed = torch.einsum(
    "ij,bjc->bic",
    H_res,
    streams,
)

branch_written = (
    H_post[None, :, None]
    * branch_output[:, None, :]
)

hyper_output = residual_mixed + branch_written

print("branch input:", branch_input.shape)
print("multi-stream output:", hyper_output.shape)


## 4. Sinkhorn-Knopp projection

mHC의 `H_res`는 non-negative이고 모든 row sum과 column sum이 1인 doubly-stochastic matrix로 제한된다. 작은 예제에서는 반복적인 row/column normalization으로 이 projection을 직접 확인한다.


In [ ]:
def sinkhorn(logits, iterations=20):
    matrix = logits.float().exp()

    for _ in range(iterations):
        matrix = matrix / matrix.sum(
            dim=-1,
            keepdim=True,
        )
        matrix = matrix / matrix.sum(
            dim=-2,
            keepdim=True,
        )

    return matrix.to(logits.dtype)


raw_matrix = torch.randn(
    2,
    num_streams,
    num_streams,
    device=device,
)

projected = sinkhorn(raw_matrix)

print(
    "row sums:",
    projected.sum(dim=-1),
)
print(
    "column sums:",
    projected.sum(dim=-2),
)


## 5. Paper-faithful tiny dynamic mHC

논문의 dynamic mHC에서 mapping을 input-dependent하게 만들 때 중요한 것은 단순 `Linear -> sigmoid`가 아니다.

작은 residual-routing 변화가 안정적으로 시작되도록

`dynamic = alpha * phi(RMSNorm(X))`

형태의 작은 learned scale `alpha`를 두고, static bias에 더한 뒤 각 manifold constraint를 적용한다.

여기서는 구조를 읽기 쉽게 하기 위해 residual streams를 flatten해서 작은 hyper-network 입력으로 사용한다. 시스템용 fused kernel과 memory optimization은 생략하지만 routing 계산 그래프는 유지한다.


In [ ]:
class TinyDynamicMHC(nn.Module):
    def __init__(
        self,
        num_streams=3,
        channels=8,
        dynamic_scale_init=1e-2,
    ):
        super().__init__()

        self.num_streams = num_streams
        self.channels = channels

        flattened_dim = num_streams * channels

        self.norm = nn.RMSNorm(flattened_dim)

        self.pre_projection = nn.Linear(
            flattened_dim,
            num_streams,
            bias=False,
        )
        self.post_projection = nn.Linear(
            flattened_dim,
            num_streams,
            bias=False,
        )
        self.res_projection = nn.Linear(
            flattened_dim,
            num_streams * num_streams,
            bias=False,
        )

        self.alpha_pre = nn.Parameter(
            torch.tensor(dynamic_scale_init)
        )
        self.alpha_post = nn.Parameter(
            torch.tensor(dynamic_scale_init)
        )
        self.alpha_res = nn.Parameter(
            torch.tensor(dynamic_scale_init)
        )

        # Static biases provide an identity-like starting point.
        self.base_pre = nn.Parameter(
            torch.ones(num_streams)
        )
        self.base_post = nn.Parameter(
            torch.zeros(num_streams)
        )

        identity = torch.eye(num_streams)
        self.base_res = nn.Parameter(
            4.0 * identity
        )

        self.branch = nn.Sequential(
            nn.Linear(channels, 2 * channels),
            nn.SiLU(),
            nn.Linear(2 * channels, channels),
        )

    def routing_maps(self, streams):
        batch_size = streams.size(0)

        flattened = streams.reshape(
            batch_size,
            -1,
        )
        normalized = self.norm(flattened)

        dynamic_pre = self.pre_projection(normalized)
        dynamic_post = self.post_projection(normalized)
        dynamic_res = self.res_projection(normalized)

        raw_pre = (
            self.base_pre[None, :]
            + self.alpha_pre * dynamic_pre
        )
        raw_post = (
            self.base_post[None, :]
            + self.alpha_post * dynamic_post
        )
        raw_res = (
            self.base_res[None, :, :]
            + self.alpha_res
            * dynamic_res.view(
                batch_size,
                self.num_streams,
                self.num_streams,
            )
        )

        H_pre = torch.sigmoid(raw_pre)
        H_post = 2.0 * torch.sigmoid(raw_post)
        H_res = sinkhorn(raw_res)

        return H_pre, H_post, H_res

    def forward(self, streams):
        H_pre, H_post, H_res = self.routing_maps(
            streams
        )

        branch_input = torch.einsum(
            "bs,bsc->bc",
            H_pre,
            streams,
        )

        branch_output = self.branch(branch_input)

        residual_mixed = torch.einsum(
            "bij,bjc->bic",
            H_res,
            streams,
        )

        branch_written = (
            H_post[:, :, None]
            * branch_output[:, None, :]
        )

        output = residual_mixed + branch_written

        return output, (H_pre, H_post, H_res)


mhc = TinyDynamicMHC().to(device)

mhc_output, (
    H_pre,
    H_post,
    H_res,
) = mhc(streams)

print("H_pre:", H_pre.shape)
print("H_post:", H_post.shape)
print("H_res:", H_res.shape)
print(
    "H_res row sums:",
    H_res.sum(dim=-1),
)
print(
    "H_res column sums:",
    H_res.sum(dim=-2),
)
print("mHC output:", mhc_output.shape)


## 6. Gradient-flow sanity check

이 노트북의 목적은 대규모 pretraining이 아니라 **논문의 routing parameterization 전체에 gradient가 실제로 흐르는지** 확인하는 것이다.


In [ ]:
target = torch.randn_like(mhc_output)

loss = F.mse_loss(
    mhc_output,
    target,
)

loss.backward()

gradient_report = {
    "alpha_pre": mhc.alpha_pre.grad.abs().item(),
    "alpha_post": mhc.alpha_post.grad.abs().item(),
    "alpha_res": mhc.alpha_res.grad.abs().item(),
    "pre_projection": (
        mhc.pre_projection.weight.grad.norm().item()
    ),
    "post_projection": (
        mhc.post_projection.weight.grad.norm().item()
    ),
    "res_projection": (
        mhc.res_projection.weight.grad.norm().item()
    ),
}

print("loss:", loss.item())

for name, value in gradient_report.items():
    print(f"{name}: {value:.6e}")


## References and provenance

**Residual / ReZero / LayerScale** — residual branch scaling 계열의 핵심 계산을 비교한다.

**Hyper-Connections** — multi-stream residual state와 `H_pre`, `H_res`, `H_post` update를 반영한다.

**mHC: Manifold-Constrained Hyper-Connections** — input-dependent routing에 작은 learned scale과 static bias를 사용하고, `H_pre = sigmoid`, `H_post = 2 sigmoid`, `H_res = Sinkhorn(...)` 제약을 적용하는 핵심 구조를 반영했다.

대규모 모델의 fused kernels, distributed training, memory-layout optimization은 이 최소구현의 범위가 아니다.
